# Dataset feasibility

This notebook uses the same reusable code as the command-line report. It measures evenly spaced subsets of SHD and DVS Gesture, generates ten-sample visual inspection sheets, and creates the SHD-specific diagnostics required by the frozen
protocol. Raw downloads are never modified.

In [1]:
import sys
from dataclasses import asdict
from pathlib import Path

from IPython.display import Image, display

repository_root = Path.cwd().resolve()
"""Absolute path to the repository root."""
if repository_root.name == "notebooks":
  repository_root = repository_root.parent
sys.path.insert(0, str(repository_root))

from src.data.feasibility import analyze_dataset, load_dataset, plot_event_samples, plot_shd_diagnostics
from src.utils.config import load_config, repository_path

config = load_config()
"""Frozen experiment configuration."""
settings = config["feasibility"]
"""Dataset feasibility settings."""
raw_path = repository_path(config["paths"]["raw_data"])
"""Directory containing immutable raw datasets."""
output_path = repository_path(config["paths"]["results"]) / "dataset_feasibility"
"""Directory receiving generated summaries and figures."""
output_path.mkdir(parents=True, exist_ok=True)

## SHD

Tonic exposes SHD events as `(t, x, p)`. The summary reports event counts, durations, class balance, disk usage, preprocessing time, input dimensionality, and an explicitly defined QRC cost proxy.

In [ ]:
shd = load_dataset("shd", raw_path, train=True)
"""SHD training dataset loaded through Tonic."""
shd_summary = analyze_dataset(
  shd,
  "shd",
  "train",
  sample_limit=settings["sample_limit"],
  temporal_bin_us=settings["temporal_bin_us"],
  shd_pooled_channels=settings["shd_pooled_channels"],
  dvs_spatial_bins=settings["dvs_spatial_bins"],
  qrc_qubits=settings["qrc_qubits"],
  qrc_circuit_depth=settings["qrc_circuit_depth"]
)
"""Measured SHD feasibility summary."""
asdict(shd_summary)

In [ ]:
plot_event_samples(shd, "shd", output_path / "shd_ten_samples.png", count=10)
plot_shd_diagnostics(shd, output_path / "shd_diagnostics.png", sample_limit=settings["sample_limit"])
display(Image(filename=output_path / "shd_ten_samples.png"))
display(Image(filename=output_path / "shd_diagnostics.png"))

## DVS Gesture

Tonic exposes DVS Gesture events as `(x, y, p, t)`. The feasibility probe pools the $128 \times 128$ sensor to $8 \times 8$ regions with separate polarities ($128$ features). This is only a cost probe; it does not establish the final
preprocessing for a deferred dataset.

In [ ]:
dvs = load_dataset("dvs", raw_path, train=True)
"""DVS Gesture training dataset loaded through Tonic."""
dvs_summary = analyze_dataset(
  dvs,
  "dvs",
  "train",
  sample_limit=settings["sample_limit"],
  temporal_bin_us=settings["temporal_bin_us"],
  shd_pooled_channels=settings["shd_pooled_channels"],
  dvs_spatial_bins=settings["dvs_spatial_bins"],
  qrc_qubits=settings["qrc_qubits"],
  qrc_circuit_depth=settings["qrc_circuit_depth"]
)
"""Measured DVS Gesture feasibility summary."""
asdict(dvs_summary)

In [ ]:
plot_event_samples(dvs, "dvs", output_path / "dvs_gesture_ten_samples.png", count=10)
display(Image(filename=output_path / "dvs_gesture_ten_samples.png"))

## Decision

The measured summaries support the decision in `dataset_decision.md`: freeze SHD with speaker-disjoint validation and defer DVS Gesture because spatial pooling and substantially greater event volume would confound the initial reservoir comparison.